In [ ]:
import os, sys, zipfile, subprocess
from pathlib import Path
from google.colab import drive

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pytorch-msssim'])

PROJECT = Path('/content/drive/MyDrive/ImageHarmonization/iHarmony4_dataset_resize_256')
DATA = Path('/content/iharmony4_256')
DATA.mkdir(parents=True, exist_ok=True)

ARCHIVES = ['HCOCO', 'HAdobe5k', 'HFlickr', 'Hday2night']
for subset in ARCHIVES:
    dest = DATA / subset
    if not (dest / '.extracted_ok').exists():
        zip_path = PROJECT / f'{subset}_256.zip'
        assert zip_path.is_file(), f'Không tìm thấy {zip_path}'
        print(f'Đang giải nén {zip_path.name}...')
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(DATA)
        (dest / '.extracted_ok').touch()

print('Dữ liệu 256x256 sẵn sàng tại:', DATA)

Đang giải nén HCOCO_256.zip...
Đang giải nén HAdobe5k_256.zip...
Đang giải nén HFlickr_256.zip...
Đang giải nén Hday2night_256.zip...
Dữ liệu 256x256 sẵn sàng tại: /content/iharmony4_256


In [ ]:
import json, random, hashlib, time, platform
from collections import defaultdict, Counter
import numpy as np
from PIL import Image
import torch
from torch import nn
import torch.nn.functional as F
from torch.nn.utils import spectral_norm
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms.functional import pil_to_tensor
from torchvision.utils import save_image
from pytorch_msssim import ssim
from tqdm.auto import tqdm

# ==============================================================================
# CẤU HÌNH NHÁNH B (+1 PatchGAN + D-Warmup + SpectralNorm, Epoch 31 -> 50)
# ==============================================================================
BASE_CKPT = PROJECT / 'runs/unet_masked_l1_50ep_all4_bs32_seed42/best_30ep.pt'
RUN = PROJECT / 'runs/unet_branchB_1patchgan_50ep_all4_bs32_seed42'
RUN.mkdir(parents=True, exist_ok=True)

CFG = dict(
    branch='B',
    seed=42,
    start_epoch=30,
    total_epochs=50,
    extra_epochs=20,
    size=256,
    batch_size=32,
    workers=2,
    base_channels=32,
    lr_G=5e-5,
    lr_D=1e-4,
    eta_min=1e-6,
    lambda_l1=100.0,
    real_label=0.9,
    n_critic_warmup_epochs=3,
    val_fraction=0.05,
    loss='masked_l1_plus_1_patchgan_balanced',
    architecture='unet_groupnorm_v1_sn_patchgan',
)

DEVICE = torch.device('cuda')
random.seed(CFG['seed'])
np.random.seed(CFG['seed'])
torch.manual_seed(CFG['seed'])
torch.cuda.manual_seed_all(CFG['seed'])
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

# 1. Lập chỉ mục & chia Train / Val / Test khớp 100% các run trước
EXTS = {'.jpg', '.jpeg', '.png'}
def make_index(folder):
    return {p.stem: p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in EXTS and not p.name.startswith('.')}

records = {'train': [], 'val': [], 'test': []}
for subset in ARCHIVES:
    subset_root = DATA / subset
    comp = make_index(subset_root / 'composite_images')
    mask = make_index(subset_root / 'masks')
    real = make_index(subset_root / 'real_images')

    subsets = {}
    for split in ('train', 'test'):
        txt = subset_root / f'{subset}_{split}.txt'
        rows = []
        for line in txt.read_text(encoding='utf-8-sig').splitlines():
            if not line.strip():
                continue
            stem = Path(line.strip().replace('\\', '/').split()[0]).stem
            parts = stem.split('_')
            source_id, mask_id = '_'.join(parts[:-2]), '_'.join(parts[:-1])
            rows.append(dict(
                subset=subset, stem=stem, group=f'{subset}/{source_id}',
                comp=str(comp[stem]), mask=str(mask[mask_id]), real=str(real[source_id]),
            ))
        subsets[split] = rows

    groups = sorted({r['group'] for r in subsets['train']})
    random.Random(CFG['seed']).shuffle(groups)
    val_groups = set(groups[:max(1, round(len(groups) * CFG['val_fraction']))])

    for r in subsets['train']:
        records['val' if r['group'] in val_groups else 'train'].append(r)
    records['test'].extend(subsets['test'])

split_ids = {s: [f"{r['subset']}/{r['stem']}" for r in rows] for s, rows in records.items()}
fingerprint = hashlib.sha256(json.dumps(split_ids, sort_keys=True).encode()).hexdigest()
print(f"Train: {len(records['train'])} | Val: {len(records['val'])} | Test: {len(records['test'])}")


class HarmonyDataset(Dataset):
    def __init__(self, rows, train=False):
        self.rows, self.train = rows, train

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        target_size = (CFG['size'], CFG['size'])
        images = []
        for key, mode in [('comp', 'RGB'), ('mask', 'L'), ('real', 'RGB')]:
            with Image.open(r[key]) as im:
                im = im.convert(mode)
                if im.size != target_size:
                    resample = Image.Resampling.NEAREST if key == 'mask' else Image.Resampling.BILINEAR
                    im = im.resize(target_size, resample)
                images.append(pil_to_tensor(im).float() / 255.)
        c, m, y = images
        m_bin = (m >= 0.5).float()
        m = m_bin if m_bin.sum() > 0 else (m > 0).float()
        if self.train and torch.rand(()) < 0.5:
            c, m, y = [x.flip(-1) for x in (c, m, y)]
        return c, m, y, r['subset']


def loader(split, epoch=0):
    g = torch.Generator().manual_seed(CFG['seed'] + epoch)
    is_train = (split == 'train')
    return DataLoader(
        HarmonyDataset(records[split], is_train),
        batch_size=CFG['batch_size'],
        shuffle=is_train,
        num_workers=CFG['workers'],
        pin_memory=True,
        drop_last=False,
        persistent_workers=(CFG['workers'] > 0),
        generator=g,
    )


# 2. Kiến trúc Generator U-Net (khớp 100% checkpoint gốc)
class Block(nn.Sequential):
    def __init__(self, cin, cout):
        super().__init__(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False),
            nn.GroupNorm(8, cout),
            nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False),
            nn.GroupNorm(8, cout),
            nn.ReLU(inplace=True),
        )


class UNet(nn.Module):
    def __init__(self, b=32):
        super().__init__()
        self.enc = nn.ModuleList([Block(4, b), Block(b, b*2), Block(b*2, b*4), Block(b*4, b*8), Block(b*8, b*16)])
        self.up = nn.ModuleList([
            nn.ConvTranspose2d(b*16, b*8, 2, 2),
            nn.ConvTranspose2d(b*8, b*4, 2, 2),
            nn.ConvTranspose2d(b*4, b*2, 2, 2),
            nn.ConvTranspose2d(b*2, b, 2, 2),
        ])
        self.dec = nn.ModuleList([Block(b*16, b*8), Block(b*8, b*4), Block(b*4, b*2), Block(b*2, b)])
        self.head = nn.Conv2d(b, 3, 1)

    def forward(self, x):
        skips = []
        for i, block in enumerate(self.enc):
            x = block(x)
            if i < 4:
                skips.append(x)
                x = F.max_pool2d(x, 2)
        for up, block, skip in zip(self.up, self.dec, reversed(skips)):
            x = block(torch.cat([up(x), skip], 1))
        return self.head(x).sigmoid()


# 3. Kiến trúc Discriminator: 1 PatchGAN 70x70 có Spectral Normalization
class PatchGANDiscriminator(nn.Module):
    def __init__(self, input_nc=4, ndf=64):
        super().__init__()
        self.net = nn.Sequential(
            spectral_norm(nn.Conv2d(input_nc, ndf, 4, stride=2, padding=1)),
            nn.LeakyReLU(0.2, inplace=True),
            spectral_norm(nn.Conv2d(ndf, ndf * 2, 4, stride=2, padding=1, bias=False)),
            nn.InstanceNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            spectral_norm(nn.Conv2d(ndf * 2, ndf * 4, 4, stride=2, padding=1, bias=False)),
            nn.InstanceNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            spectral_norm(nn.Conv2d(ndf * 4, ndf * 8, 4, stride=1, padding=1, bias=False)),
            nn.InstanceNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),
            spectral_norm(nn.Conv2d(ndf * 8, 1, 4, stride=1, padding=1)),
        )

    def forward(self, x):
        return self.net(x)


def masked_l1_loss(pred, target, mask, eps=1e-6):
    diff = torch.abs(pred - target) * mask
    per_img = diff.sum(dim=(1, 2, 3)) / (3.0 * mask.sum(dim=(1, 2, 3)).clamp_min(eps))
    return per_img.mean()


@torch.inference_mode()
def evaluate(loader_, save_to=None, include_ssim=False):
    model.eval()
    sums = defaultdict(Counter)
    for c, m, y, subsets in tqdm(loader_, desc='evaluate', leave=False):
        c, m, y = [x.to(DEVICE, non_blocking=True) for x in (c, m, y)]
        with torch.autocast('cuda', dtype=torch.float16):
            raw = model(torch.cat([c, m], 1))
        raw = raw.float()
        out = raw * m + c * (1 - m)
        err = (out - y).square()
        mse = err.mean((1, 2, 3))
        fmse = (err * m).sum((1, 2, 3)) / (3 * m.sum((1, 2, 3)).clamp_min(1.0))

        values = dict(
            raw_L1=(raw - y).abs().mean((1, 2, 3)),
            MSE=mse * 255**2,
            fMSE=fmse * 255**2,
            PSNR=-10 * torch.log10(mse.clamp_min(1e-12)),
            fPSNR=-10 * torch.log10(fmse.clamp_min(1e-12)),
        )
        if include_ssim:
            values['SSIM'] = ssim(out, y, data_range=1, size_average=False)

        for i, subset in enumerate(subsets):
            for label in ('all', subset):
                sums[label]['n'] += 1
                for k, v in values.items():
                    sums[label][k] += v[i].item()

        if save_to is not None:
            panels = torch.stack([c, m.expand_as(c), raw, out, y], 1).flatten(0, 1)
            save_image(panels[:30], str(save_to), nrow=5)
            save_to = None

    return {s: {k: (v if k == 'n' else v / d['n']) for k, v in d.items()} for s, d in sums.items()}


def atomic_save(obj, target):
    tmp = target.with_name(target.name + '.tmp')
    torch.save(obj, tmp)
    os.replace(tmp, target)

Train: 62463 | Val: 3279 | Test: 7404


In [ ]:
# ==============================================================================
# KHỞI TẠO & TRAIN LOOP NHÁNH B
# ==============================================================================
model = UNet(CFG['base_channels']).to(DEVICE)
netD = PatchGANDiscriminator(input_nc=4, ndf=64).to(DEVICE)

opt_G = torch.optim.Adam(model.parameters(), lr=CFG['lr_G'], betas=(0.5, 0.999))
opt_D = torch.optim.Adam(netD.parameters(), lr=CFG['lr_D'], betas=(0.5, 0.999))

# G chỉ bắt đầu cập nhật từ Epoch 32 -> 50 (19 epochs), D chạy cả 20 epochs (31 -> 50)
sched_G = torch.optim.lr_scheduler.CosineAnnealingLR(opt_G, T_max=CFG['extra_epochs'] - 1, eta_min=CFG['eta_min'])
sched_D = torch.optim.lr_scheduler.CosineAnnealingLR(opt_D, T_max=CFG['extra_epochs'], eta_min=CFG['eta_min'])

scaler_G = torch.amp.GradScaler('cuda')
scaler_D = torch.amp.GradScaler('cuda')

start_ep = 31
best_overall = float('inf')
best_gan = float('inf')
history = []

if (RUN / 'last.pt').exists():
    ck = torch.load(RUN / 'last.pt', map_location='cpu', weights_only=False)
    assert ck['cfg'] == CFG and ck['fingerprint'] == fingerprint, 'Config/data changed; use new RUN'
    model.load_state_dict(ck['model'])
    netD.load_state_dict(ck['netD'])
    opt_G.load_state_dict(ck['opt_G'])
    opt_D.load_state_dict(ck['opt_D'])
    for opt in (opt_G, opt_D):
        for state in opt.state.values():
            for k, v in state.items():
                if torch.is_tensor(v):
                    state[k] = v.to(DEVICE)
    sched_G.load_state_dict(ck['sched_G'])
    sched_D.load_state_dict(ck['sched_D'])
    scaler_G.load_state_dict(ck['scaler_G'])
    scaler_D.load_state_dict(ck['scaler_D'])
    start_ep = ck['epoch'] + 1
    best_overall = ck['best']
    best_gan = ck.get('best_gan', best_overall)
    history = ck['history']
    torch.set_rng_state(ck['torch_rng'])
    torch.cuda.set_rng_state_all(ck['cuda_rng'])
    print(f"Resume Branch B từ Epoch {start_ep}/50 (best_overall={best_overall:.2f}, best_gan={best_gan:.2f})")
else:
    assert BASE_CKPT.is_file(), f"Không tìm thấy checkpoint gốc: {BASE_CKPT}"
    base_ck = torch.load(BASE_CKPT, map_location='cpu', weights_only=False)
    assert base_ck['fingerprint'] == fingerprint, 'Split fingerprint không khớp!'
    model.load_state_dict(base_ck['model'])
    best_overall = base_ck['val']['all']['fMSE']
    print(f"Khởi tạo Branch B từ {BASE_CKPT} (Epoch {base_ck.get('epoch', 30)}, val fMSE={best_overall:.2f})")

(RUN / 'config.json').write_text(json.dumps(dict(
    cfg=CFG,
    fingerprint=fingerprint,
    source_ckpt=str(BASE_CKPT),
    gpu=torch.cuda.get_device_name(),
    torch=torch.__version__,
    python=platform.python_version(),
), indent=2))

for ep in range(start_ep, CFG['total_epochs'] + 1):
    torch.cuda.synchronize()
    tic = time.perf_counter()
    torch.cuda.reset_peak_memory_stats()

    is_d_warmup = (ep == 31)
    model.eval() if is_d_warmup else model.train()
    netD.train()

    tot_l1, tot_gan, tot_d, count = 0.0, 0.0, 0.0, 0
    n_critic = 2 if ep < 31 + CFG['n_critic_warmup_epochs'] else 1
    phase_str = "D-Warmup" if is_d_warmup else f"nD={n_critic}"
    bar = tqdm(loader('train', ep), desc=f'Branch B | Ep {ep}/50 ({phase_str})')

    for c, m, y, _ in bar:
        c, m, y = [x.to(DEVICE, non_blocking=True) for x in (c, m, y)]
        inp = torch.cat([c, m], dim=1)

        # 1. Cập nhật Discriminator D (n_critic=2 trong 3 epoch đầu 31-33, sau đó 1:1)
        for _ in range(n_critic):
            opt_D.zero_grad(set_to_none=True)
            with torch.autocast('cuda', dtype=torch.float16):
                with torch.no_grad():
                    raw_fake = model(inp)
                    comp_fake = raw_fake * m + c * (1.0 - m)
                pred_real = netD(torch.cat([y, m], dim=1))
                pred_fake = netD(torch.cat([comp_fake.detach(), m], dim=1))
                loss_D = 0.5 * (
                    F.mse_loss(pred_real, torch.full_like(pred_real, CFG['real_label'])) +
                    F.mse_loss(pred_fake, torch.zeros_like(pred_fake))
                )
            if not torch.isfinite(loss_D):
                raise RuntimeError('Non-finite loss_D; inspect data/AMP')

            scaler_D.scale(loss_D).backward()
            scaler_D.unscale_(opt_D)
            nn.utils.clip_grad_norm_(netD.parameters(), 1.0)
            scaler_D.step(opt_D)
            scaler_D.update()

        # 2. Cập nhật Generator G (Epoch 31 đóng băng G để D warm-up; từ Epoch 32 bật GAN loss)
        if is_d_warmup:
            with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):
                l1_val = masked_l1_loss(raw_fake, y, m)
            gan_val = 0.0
        else:
            opt_G.zero_grad(set_to_none=True)
            with torch.autocast('cuda', dtype=torch.float16):
                raw = model(inp)
                comp_out = raw * m + c * (1.0 - m)
                l1_val = masked_l1_loss(raw, y, m)
                pred_fake_g = netD(torch.cat([comp_out, m], dim=1))
                gan_loss = F.mse_loss(pred_fake_g, torch.ones_like(pred_fake_g))
                loss_G = CFG['lambda_l1'] * l1_val + gan_loss
                gan_val = gan_loss.item()

            if not torch.isfinite(loss_G):
                raise RuntimeError('Non-finite loss_G; inspect data/AMP')

            scaler_G.scale(loss_G).backward()
            scaler_G.unscale_(opt_G)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler_G.step(opt_G)
            scaler_G.update()

        bs = c.shape[0]
        tot_l1 += l1_val.item() * bs
        tot_gan += gan_val * bs
        tot_d += loss_D.item() * bs
        count += bs
        bar.set_postfix(L1=f'{tot_l1/count:.5f}', GAN=f'{tot_gan/count:.4f}', D=f'{tot_d/count:.4f}')

    assert count == len(records['train']), f'Not all training samples were seen ({count} vs {len(records["train"])})'

    scores = evaluate(loader('val'), save_to=RUN / f'val_{ep:03d}.png')
    cur_fmse = scores['all']['fMSE']

    improved_overall = cur_fmse < best_overall
    best_overall = min(best_overall, cur_fmse)

    lr_g_used = opt_G.param_groups[0]['lr']
    lr_d_used = opt_D.param_groups[0]['lr']
    sched_D.step()
    if not is_d_warmup:
        sched_G.step()
    torch.cuda.synchronize()

    row = dict(
        branch='B',
        epoch=ep,
        train_L1=tot_l1 / count,
        train_GAN=tot_gan / count,
        train_D=tot_d / count,
        val=scores,
        lr_G=lr_g_used,
        lr_D=lr_d_used,
        seconds=time.perf_counter() - tic,
        peak_VRAM_GB=torch.cuda.max_memory_allocated() / 2**30,
    )
    history.append(row)

    payload = dict(model=model.state_dict(), netD=netD.state_dict(), cfg=CFG, epoch=ep, fingerprint=fingerprint, val=scores)

    # Lưu best vượt qua cả mốc 30ep gốc
    if improved_overall:
        atomic_save(payload, RUN / 'best.pt')

    # Lưu best riêng trong giai đoạn đã bật GAN (Ep 32 -> 50)
    if not is_d_warmup and cur_fmse < best_gan:
        best_gan = cur_fmse
        atomic_save(payload, RUN / 'best_gan.pt')

    atomic_save(dict(
        epoch=ep,
        model=model.state_dict(),
        netD=netD.state_dict(),
        opt_G=opt_G.state_dict(),
        opt_D=opt_D.state_dict(),
        sched_G=sched_G.state_dict(),
        sched_D=sched_D.state_dict(),
        scaler_G=scaler_G.state_dict(),
        scaler_D=scaler_D.state_dict(),
        cfg=CFG,
        fingerprint=fingerprint,
        best=best_overall,
        best_gan=best_gan,
        history=history,
        torch_rng=torch.get_rng_state(),
        cuda_rng=torch.cuda.get_rng_state_all(),
    ), RUN / 'last.pt')
    (RUN / 'history.json').write_text(json.dumps(history, indent=2))
    print(json.dumps(row, ensure_ascii=False))

print(f'Done Branch B! Best overall fMSE: {best_overall:.2f} | Best GAN-phase (Ep 32-50) fMSE: {best_gan:.2f}')

Khởi tạo Branch B từ /content/drive/MyDrive/ImageHarmonization/iHarmony4_dataset_resize_256/runs/unet_masked_l1_50ep_all4_bs32_seed42/best_30ep.pt (Epoch 29, val fMSE=397.88)


Branch B | Ep 31/50 (D-Warmup):   0%|          | 0/1952 [00:00<?, ?it/s]

evaluate:   0%|          | 0/103 [00:00<?, ?it/s]

{"branch": "B", "epoch": 31, "train_L1": 0.05030149165650969, "train_GAN": 0.0, "train_D": 0.22105595626399432, "val": {"all": {"n": 3279, "raw_L1": 0.17709338885185813, "MSE": 40.5436619793398, "fMSE": 397.87979494314726, "PSNR": 36.00872848920249, "fPSNR": 24.34944521591917}, "HCOCO": {"n": 1924, "raw_L1": 0.17464156051141186, "MSE": 23.140737493357886, "fMSE": 369.6356207813889, "PSNR": 37.233579558295176, "fPSNR": 24.057863148235235}, "HAdobe5k": {"n": 974, "raw_L1": 0.1866478298999201, "MSE": 39.29364007374834, "fMSE": 324.07418924729194, "PSNR": 35.99174632281983, "fPSNR": 26.238237727594083}, "HFlickr": {"n": 368, "raw_L1": 0.165660142817575, "MSE": 128.65199368932971, "fMSE": 710.9841192908909, "PSNR": 29.943084141482476, "fPSNR": 21.110519061917845}, "Hday2night": {"n": 13, "raw_L1": 0.14776467245358688, "MSE": 215.68842950234045, "fMSE": 1244.4997699444111, "PSNR": 27.707520264845627, "fPSNR": 17.675969563997707}}, "lr_G": 5e-05, "lr_D": 0.0001, "seconds": 1129.86094571, "pea

Branch B | Ep 32/50 (nD=2):   0%|          | 0/1952 [00:00<?, ?it/s]

evaluate:   0%|          | 0/103 [00:00<?, ?it/s]

... (60 outputs left out)

Branch B | Ep 50/50 (nD=1):   0%|          | 0/1952 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b71ac859580>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1706, in __del__
    self._shutdown_workers()Exception ignored in: 
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1675, in _shutdown_workers
    <function _MultiProcessingDataLoaderIter.__del__ at 0x7b71ac859580>if w.is_alive():

Traceback (most recent call last):
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1706, in __del__
        assert self._parent_pid == os.getpid(), 'can only test a child process'
self._shutdown_workers()
AssertionError  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1675, in _shutdown_workers
:     can only test a child process
if w.is_alive():
  File "/usr/lib/

evaluate:   0%|          | 0/103 [00:00<?, ?it/s]

{"branch": "B", "epoch": 50, "train_L1": 0.046777209195183904, "train_GAN": 0.3948989744878263, "train_D": 0.17270451611989154, "val": {"all": {"n": 3279, "raw_L1": 0.18229852679499986, "MSE": 40.17927428617387, "fMSE": 393.5835171019242, "PSNR": 36.03658968386544, "fPSNR": 24.386846647352733}, "HCOCO": {"n": 1924, "raw_L1": 0.18097040193492311, "MSE": 22.439380022071752, "fMSE": 360.6193863666479, "PSNR": 37.30044669867058, "fPSNR": 24.134548261606767}, "HAdobe5k": {"n": 974, "raw_L1": 0.1892191531324044, "MSE": 40.0656323007, "fMSE": 328.2200599588163, "PSNR": 35.94476694150138, "fPSNR": 26.206810019099493}, "HFlickr": {"n": 368, "raw_L1": 0.17207106150200832, "MSE": 127.652148730405, "fMSE": 704.9256367994392, "PSNR": 29.96157887707586, "fPSNR": 21.1223256095596}, "Hday2night": {"n": 13, "raw_L1": 0.1498623272547355, "MSE": 198.04281755594107, "fMSE": 1356.1292665921724, "PSNR": 27.834930566641, "fPSNR": 17.78080910902757}}, "lr_G": 1.334148066633303e-06, "lr_D": 1.609427140540686e-